# Chapter 6.4 & 6.5 Tasks
## Logging, Monitoring, Error Handling & Data Storage Strategies

**Task 1**: Load data from CSV (Kaggle), transform, write to database, log each event  
**Task 2**: Load data from API, transform, write to database with exception handling  
**Task 3**: Send email notification after loading to database

In [1]:
# Import required libraries
import pandas as pd
import sqlite3
import logging
import requests
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime
import os

print("Libraries imported successfully!")

Libraries imported successfully!


## Task 1: Load CSV Data, Transform, Write to Database with Logging

- Load data from Kaggle CSV file
- Apply transformations (cleaning, handling missing values)
- Write to SQLite database
- Log each event to a log file

In [2]:
# Configure logging
log_file = 'pipeline_logs.log'

# Create a logger
logger = logging.getLogger('DataPipeline')
logger.setLevel(logging.DEBUG)

# Remove existing handlers to avoid duplicates
if logger.handlers:
    logger.handlers.clear()

# Create file handler for logging to file
file_handler = logging.FileHandler(log_file, mode='w')
file_handler.setLevel(logging.DEBUG)

# Create console handler for logging to console
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)

# Create formatter and add to handlers
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

# Add handlers to logger
logger.addHandler(file_handler)
logger.addHandler(console_handler)

logger.info("Logging configured successfully!")

2026-02-12 12:03:16,475 - DataPipeline - INFO - Logging configured successfully!


In [3]:
# Task 1: Load CSV, Transform, and Write to Database with Logging

def load_csv_data(file_path):
    """Load data from CSV file"""
    logger.info(f"Starting to load CSV data from: {file_path}")
    try:
        df = pd.read_csv(file_path)
        logger.info(f"Successfully loaded {len(df)} rows and {len(df.columns)} columns")
        logger.debug(f"Columns: {list(df.columns)}")
        return df
    except FileNotFoundError as e:
        logger.error(f"File not found: {file_path}")
        raise
    except Exception as e:
        logger.error(f"Error loading CSV: {str(e)}")
        raise

def transform_data(df):
    """Transform and clean the data"""
    logger.info("Starting data transformation...")
    
    initial_rows = len(df)
    
    # Log missing values before transformation
    missing_before = df.isnull().sum().sum()
    logger.info(f"Total missing values before transformation: {missing_before}")
    
    # Handle missing values
    # Fill numeric columns with median
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
    for col in numeric_cols:
        missing_count = df[col].isnull().sum()
        if missing_count > 0:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            logger.debug(f"Filled {missing_count} missing values in '{col}' with median: {median_val}")
    
    # Fill categorical columns with mode
    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        missing_count = df[col].isnull().sum()
        if missing_count > 0:
            mode_val = df[col].mode()[0] if not df[col].mode().empty else 'Unknown'
            df[col].fillna(mode_val, inplace=True)
            logger.debug(f"Filled {missing_count} missing values in '{col}' with mode: {mode_val}")
    
    # Log missing values after transformation
    missing_after = df.isnull().sum().sum()
    logger.info(f"Total missing values after transformation: {missing_after}")
    
    # Remove duplicates
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        df.drop_duplicates(inplace=True)
        logger.info(f"Removed {duplicates} duplicate rows")
    
    # Add transformation timestamp
    df['processed_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    logger.info(f"Added processed_at timestamp column")
    
    final_rows = len(df)
    logger.info(f"Transformation complete. Rows: {initial_rows} -> {final_rows}")
    
    return df

def write_to_database(df, db_name, table_name):
    """Write DataFrame to SQLite database"""
    logger.info(f"Starting to write data to database: {db_name}, table: {table_name}")
    try:
        conn = sqlite3.connect(db_name)
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        
        # Verify the write
        cursor = conn.cursor()
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cursor.fetchone()[0]
        
        conn.close()
        logger.info(f"Successfully wrote {count} rows to '{table_name}' table in '{db_name}'")
        return count
    except Exception as e:
        logger.error(f"Error writing to database: {str(e)}")
        raise

# Execute Task 1
csv_file = r'c:\Users\mausa\Desktop\5thsem\adv_py\lab4\retail_store_sales.csv'
db_name = 'data_pipeline.db'
table_name = 'retail_sales'

logger.info("=" * 50)
logger.info("TASK 1: CSV to Database Pipeline Started")
logger.info("=" * 50)

# Load data
df_csv = load_csv_data(csv_file)
print(f"\nOriginal Data Shape: {df_csv.shape}")
print(df_csv.head())

# Transform data
df_transformed = transform_data(df_csv.copy())
print(f"\nTransformed Data Shape: {df_transformed.shape}")

# Write to database
rows_written = write_to_database(df_transformed, db_name, table_name)

logger.info("TASK 1: Pipeline completed successfully!")
print(f"\n✅ Task 1 Complete: {rows_written} rows written to database")

2026-02-12 12:03:16,539 - DataPipeline - INFO - ==================================================
2026-02-12 12:03:16,543 - DataPipeline - INFO - TASK 1: CSV to Database Pipeline Started
2026-02-12 12:03:16,547 - DataPipeline - INFO - ==================================================
2026-02-12 12:03:16,552 - DataPipeline - INFO - Starting to load CSV data from: c:\Users\mausa\Desktop\5thsem\adv_py\lab4\retail_store_sales.csv
2026-02-12 12:03:16,683 - DataPipeline - INFO - Successfully loaded 12575 rows and 11 columns
2026-02-12 12:03:16,716 - DataPipeline - INFO - Starting data transformation...
2026-02-12 12:03:16,731 - DataPipeline - INFO - Total missing values before transformation: 7229
C:\Users\mausa\AppData\Local\Temp\ipykernel_17648\2536657522.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediat


Original Data Shape: (12575, 11)
  Transaction ID Customer ID       Category          Item  Price Per Unit  \
0    TXN_6867343     CUST_09     Patisserie   Item_10_PAT            18.5   
1    TXN_3731986     CUST_22  Milk Products  Item_17_MILK            29.0   
2    TXN_9303719     CUST_02       Butchers   Item_12_BUT            21.5   
3    TXN_9458126     CUST_06      Beverages   Item_16_BEV            27.5   
4    TXN_4575373     CUST_05           Food   Item_6_FOOD            12.5   

   Quantity  Total Spent  Payment Method Location Transaction Date  \
0      10.0        185.0  Digital Wallet   Online       2024-04-08   
1       9.0        261.0  Digital Wallet   Online       2023-07-23   
2       2.0         43.0     Credit Card   Online       2022-10-05   
3       9.0        247.5     Credit Card   Online       2022-05-07   
4       7.0         87.5  Digital Wallet   Online       2022-10-02   

  Discount Applied  
0             True  
1             True  
2            False 

2026-02-12 12:03:16,934 - DataPipeline - INFO - Successfully wrote 12575 rows to 'retail_sales' table in 'data_pipeline.db'
2026-02-12 12:03:16,935 - DataPipeline - INFO - TASK 1: Pipeline completed successfully!



✅ Task 1 Complete: 12575 rows written to database


## Task 2: Load Data from API, Transform, Write to Database with Exception Handling

- Fetch data from a public API
- Transform the data
- Write to SQLite database
- Implement proper exception handling for network issues, API errors, etc.

In [4]:
# Task 2: Load from API, Transform, Write to Database with Exception Handling

class APIError(Exception):
    """Custom exception for API errors"""
    pass

class DataTransformationError(Exception):
    """Custom exception for data transformation errors"""
    pass

class DatabaseError(Exception):
    """Custom exception for database errors"""
    pass

def fetch_data_from_api(api_url, timeout=30):
    """
    Fetch data from API with comprehensive exception handling
    """
    logger.info(f"Fetching data from API: {api_url}")
    
    try:
        response = requests.get(api_url, timeout=timeout)
        
        # Check HTTP status code
        response.raise_for_status()
        
        data = response.json()
        logger.info(f"Successfully fetched data from API")
        
        return data
        
    except requests.exceptions.Timeout:
        logger.error(f"API request timed out after {timeout} seconds")
        raise APIError(f"Request timed out after {timeout} seconds")
        
    except requests.exceptions.ConnectionError:
        logger.error("Failed to connect to API - Network error")
        raise APIError("Network connection error")
        
    except requests.exceptions.HTTPError as e:
        logger.error(f"HTTP error occurred: {e.response.status_code}")
        raise APIError(f"HTTP error: {e.response.status_code}")
        
    except requests.exceptions.JSONDecodeError:
        logger.error("Failed to parse JSON response")
        raise APIError("Invalid JSON response from API")
        
    except Exception as e:
        logger.error(f"Unexpected error fetching data: {str(e)}")
        raise APIError(f"Unexpected error: {str(e)}")

def transform_api_data(data):
    """
    Transform API data with exception handling
    """
    logger.info("Starting API data transformation...")
    
    try:
        # Check if data is empty
        if not data:
            raise DataTransformationError("Received empty data from API")
        
        # Convert to DataFrame
        if isinstance(data, list):
            df = pd.DataFrame(data)
        elif isinstance(data, dict):
            # Handle nested JSON structures
            if 'results' in data:
                df = pd.DataFrame(data['results'])
            elif 'data' in data:
                df = pd.DataFrame(data['data'])
            else:
                df = pd.DataFrame([data])
        else:
            raise DataTransformationError(f"Unexpected data type: {type(data)}")
        
        if df.empty:
            raise DataTransformationError("DataFrame is empty after conversion")
        
        logger.info(f"Converted data to DataFrame: {len(df)} rows, {len(df.columns)} columns")
        
        # Clean column names (remove special characters, lowercase)
        df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('[^a-z0-9_]', '', regex=True)
        logger.debug(f"Cleaned column names: {list(df.columns)}")
        
        # Add metadata
        df['api_fetch_timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        logger.info("API data transformation completed successfully")
        return df
        
    except DataTransformationError:
        raise
    except Exception as e:
        logger.error(f"Error during transformation: {str(e)}")
        raise DataTransformationError(f"Transformation failed: {str(e)}")

def write_api_data_to_database(df, db_name, table_name):
    """
    Write API data to database with exception handling
    """
    logger.info(f"Writing API data to database: {db_name}, table: {table_name}")
    
    try:
        conn = sqlite3.connect(db_name)
        
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        
        # Verify write operation
        cursor = conn.cursor()
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cursor.fetchone()[0]
        
        if count != len(df):
            raise DatabaseError(f"Row count mismatch: expected {len(df)}, got {count}")
        
        conn.close()
        logger.info(f"Successfully wrote {count} rows to '{table_name}' table")
        return count
        
    except sqlite3.Error as e:
        logger.error(f"SQLite error: {str(e)}")
        raise DatabaseError(f"Database error: {str(e)}")
    except Exception as e:
        logger.error(f"Error writing to database: {str(e)}")
        raise DatabaseError(f"Write failed: {str(e)}")

# Execute Task 2
logger.info("=" * 50)
logger.info("TASK 2: API to Database Pipeline Started")
logger.info("=" * 50)

# Using JSONPlaceholder API (free fake API for testing)
api_url = "https://jsonplaceholder.typicode.com/posts"
api_table_name = "api_posts"

try:
    # Fetch data from API
    api_data = fetch_data_from_api(api_url)
    print(f"\n📥 Fetched {len(api_data)} records from API")
    
    # Transform data
    df_api = transform_api_data(api_data)
    print(f"\n📊 Transformed Data Shape: {df_api.shape}")
    print(df_api.head())
    
    # Write to database
    rows = write_api_data_to_database(df_api, db_name, api_table_name)
    
    logger.info("TASK 2: Pipeline completed successfully!")
    print(f"\n✅ Task 2 Complete: {rows} rows written to database")
    
except APIError as e:
    logger.critical(f"API Error: {str(e)}")
    print(f"❌ API Error: {e}")
    
except DataTransformationError as e:
    logger.critical(f"Transformation Error: {str(e)}")
    print(f"❌ Transformation Error: {e}")
    
except DatabaseError as e:
    logger.critical(f"Database Error: {str(e)}")
    print(f"❌ Database Error: {e}")
    
except Exception as e:
    logger.critical(f"Unexpected Error: {str(e)}")
    print(f"❌ Unexpected Error: {e}")

2026-02-12 12:03:16,998 - DataPipeline - INFO - ==================================================
2026-02-12 12:03:17,000 - DataPipeline - INFO - TASK 2: API to Database Pipeline Started
2026-02-12 12:03:17,003 - DataPipeline - INFO - ==================================================
2026-02-12 12:03:17,009 - DataPipeline - INFO - Fetching data from API: https://jsonplaceholder.typicode.com/posts
2026-02-12 12:03:18,404 - DataPipeline - INFO - Successfully fetched data from API
2026-02-12 12:03:18,414 - DataPipeline - INFO - Starting API data transformation...
2026-02-12 12:03:18,424 - DataPipeline - INFO - Converted data to DataFrame: 100 rows, 4 columns
2026-02-12 12:03:18,440 - DataPipeline - INFO - API data transformation completed successfully
2026-02-12 12:03:18,456 - DataPipeline - INFO - Writing API data to database: data_pipeline.db, table: api_posts



📥 Fetched 100 records from API

📊 Transformed Data Shape: (100, 5)
   userid  id                                              title  \
0       1   1  sunt aut facere repellat provident occaecati e...   
1       1   2                                       qui est esse   
2       1   3  ea molestias quasi exercitationem repellat qui...   
3       1   4                               eum et est occaecati   
4       1   5                                 nesciunt quas odio   

                                                body  api_fetch_timestamp  
0  quia et suscipit\nsuscipit recusandae consequu...  2026-02-12 12:03:18  
1  est rerum tempore vitae\nsequi sint nihil repr...  2026-02-12 12:03:18  
2  et iusto sed quo iure\nvoluptatem occaecati om...  2026-02-12 12:03:18  
3  ullam et saepe reiciendis voluptatem adipisci\...  2026-02-12 12:03:18  
4  repudiandae veniam quaerat sunt sed\nalias aut...  2026-02-12 12:03:18  


2026-02-12 12:03:18,624 - DataPipeline - INFO - Successfully wrote 100 rows to 'api_posts' table
2026-02-12 12:03:18,628 - DataPipeline - INFO - TASK 2: Pipeline completed successfully!



✅ Task 2 Complete: 100 rows written to database


## Task 3: Send Email Notification After Loading to Database

- Complete pipeline with email notification
- Send success/failure notification to specified email
- Include summary of pipeline execution

In [5]:
# Task 3: Email Notification Function

def send_email_notification(recipient_email, subject, body, sender_email=None, sender_password=None):
    """
    Send email notification after database operations
    
    Note: For Gmail, you need to:
    1. Enable 2-Step Verification
    2. Generate an App Password at https://myaccount.google.com/apppasswords
    3. Use the App Password instead of your regular password
    """
    logger.info(f"Preparing to send email notification to: {recipient_email}")
    
    try:
        # Create message
        msg = MIMEMultipart()
        msg['From'] = sender_email if sender_email else "your_email@gmail.com"
        msg['To'] = recipient_email
        msg['Subject'] = subject
        
        # Attach body
        msg.attach(MIMEText(body, 'html'))
        
        # Connect to Gmail SMTP server
        logger.info("Connecting to SMTP server...")
        
        if sender_email and sender_password:
            server = smtplib.SMTP('smtp.gmail.com', 587)
            server.starttls()
            server.login(sender_email, sender_password)
            
            # Send email
            server.send_message(msg)
            server.quit()
            
            logger.info(f"Email sent successfully to {recipient_email}")
            return True
        else:
            logger.warning("Email credentials not provided - Email simulation only")
            print("\n📧 Email Simulation (credentials not provided):")
            print(f"   To: {recipient_email}")
            print(f"   Subject: {subject}")
            print(f"   Body preview: {body[:200]}...")
            return False
            
    except smtplib.SMTPAuthenticationError:
        logger.error("Email authentication failed - check credentials")
        raise Exception("Authentication failed: Check email credentials or use App Password")
        
    except smtplib.SMTPException as e:
        logger.error(f"SMTP error: {str(e)}")
        raise Exception(f"SMTP error: {str(e)}")
        
    except Exception as e:
        logger.error(f"Failed to send email: {str(e)}")
        raise

In [6]:
# Complete Pipeline with Email Notification

def run_complete_pipeline_with_notification():
    """
    Run the complete data pipeline and send email notification
    """
    logger.info("=" * 50)
    logger.info("TASK 3: Complete Pipeline with Email Notification")
    logger.info("=" * 50)
    
    pipeline_status = {
        'csv_status': 'Not Started',
        'api_status': 'Not Started',
        'csv_rows': 0,
        'api_rows': 0,
        'errors': []
    }
    
    start_time = datetime.now()
    
    # Step 1: CSV Pipeline
    try:
        logger.info("Step 1: Processing CSV data...")
        df_csv = load_csv_data(csv_file)
        df_transformed = transform_data(df_csv.copy())
        csv_rows = write_to_database(df_transformed, db_name, 'retail_sales_final')
        pipeline_status['csv_status'] = 'Success'
        pipeline_status['csv_rows'] = csv_rows
        logger.info("CSV pipeline completed successfully")
    except Exception as e:
        pipeline_status['csv_status'] = 'Failed'
        pipeline_status['errors'].append(f"CSV Error: {str(e)}")
        logger.error(f"CSV pipeline failed: {str(e)}")
    
    # Step 2: API Pipeline
    try:
        logger.info("Step 2: Processing API data...")
        api_data = fetch_data_from_api("https://jsonplaceholder.typicode.com/users")
        df_api = transform_api_data(api_data)
        api_rows = write_api_data_to_database(df_api, db_name, 'api_users_final')
        pipeline_status['api_status'] = 'Success'
        pipeline_status['api_rows'] = api_rows
        logger.info("API pipeline completed successfully")
    except Exception as e:
        pipeline_status['api_status'] = 'Failed'
        pipeline_status['errors'].append(f"API Error: {str(e)}")
        logger.error(f"API pipeline failed: {str(e)}")
    
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    # Prepare email content
    overall_status = "SUCCESS" if not pipeline_status['errors'] else "COMPLETED WITH ERRORS"
    
    email_subject = f"Data Pipeline Report - {overall_status} - {datetime.now().strftime('%Y-%m-%d %H:%M')}"
    
    email_body = f"""
    <html>
    <body>
    <h2>🔄 Data Pipeline Execution Report</h2>
    
    <h3>📊 Summary</h3>
    <ul>
        <li><strong>Status:</strong> {overall_status}</li>
        <li><strong>Execution Time:</strong> {start_time.strftime('%Y-%m-%d %H:%M:%S')}</li>
        <li><strong>Duration:</strong> {duration:.2f} seconds</li>
        <li><strong>Database:</strong> {db_name}</li>
    </ul>
    
    <h3>📁 CSV Pipeline (Task 1)</h3>
    <ul>
        <li><strong>Status:</strong> {pipeline_status['csv_status']}</li>
        <li><strong>Source:</strong> {os.path.basename(csv_file)}</li>
        <li><strong>Rows Written:</strong> {pipeline_status['csv_rows']}</li>
    </ul>
    
    <h3>🌐 API Pipeline (Task 2)</h3>
    <ul>
        <li><strong>Status:</strong> {pipeline_status['api_status']}</li>
        <li><strong>Source:</strong> JSONPlaceholder API</li>
        <li><strong>Rows Written:</strong> {pipeline_status['api_rows']}</li>
    </ul>
    
    {"<h3>⚠️ Errors</h3><ul>" + "".join([f"<li>{e}</li>" for e in pipeline_status['errors']]) + "</ul>" if pipeline_status['errors'] else ""}
    
    <hr>
    <p><em>This is an automated notification from the Data Pipeline System.</em></p>
    </body>
    </html>
    """
    
    # Send email notification
    recipient_email = "abiralsujakhu121@gmail.com"
    
    logger.info(f"Sending notification email to {recipient_email}")
    
    # Note: To actually send emails, uncomment and provide your credentials
    # SENDER_EMAIL = "your_email@gmail.com"
    # SENDER_PASSWORD = "your_app_password"  # Use Gmail App Password
    
    # For demonstration (without credentials):
    send_email_notification(
        recipient_email=recipient_email,
        subject=email_subject,
        body=email_body
        # sender_email=SENDER_EMAIL,
        # sender_password=SENDER_PASSWORD
    )
    
    return pipeline_status

# Execute the complete pipeline
print("\n" + "="*60)
print("RUNNING COMPLETE PIPELINE WITH EMAIL NOTIFICATION")
print("="*60)

result = run_complete_pipeline_with_notification()

print("\n" + "="*60)
print("PIPELINE EXECUTION SUMMARY")
print("="*60)
print(f"CSV Pipeline: {result['csv_status']} ({result['csv_rows']} rows)")
print(f"API Pipeline: {result['api_status']} ({result['api_rows']} rows)")
if result['errors']:
    print(f"Errors: {len(result['errors'])}")
    for err in result['errors']:
        print(f"  - {err}")
print("="*60)

2026-02-12 12:03:18,744 - DataPipeline - INFO - ==================================================
2026-02-12 12:03:18,747 - DataPipeline - INFO - TASK 3: Complete Pipeline with Email Notification
2026-02-12 12:03:18,756 - DataPipeline - INFO - ==================================================
2026-02-12 12:03:18,761 - DataPipeline - INFO - Step 1: Processing CSV data...
2026-02-12 12:03:18,764 - DataPipeline - INFO - Starting to load CSV data from: c:\Users\mausa\Desktop\5thsem\adv_py\lab4\retail_store_sales.csv



RUNNING COMPLETE PIPELINE WITH EMAIL NOTIFICATION


2026-02-12 12:03:18,867 - DataPipeline - INFO - Successfully loaded 12575 rows and 11 columns
2026-02-12 12:03:18,874 - DataPipeline - INFO - Starting data transformation...
2026-02-12 12:03:18,890 - DataPipeline - INFO - Total missing values before transformation: 7229
C:\Users\mausa\AppData\Local\Temp\ipykernel_17648\2536657522.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)
C:\Users\mausa\AppData\Local\Temp\ipykernel_17648\2536657522.py:35: FutureWarning: A value is trying to be


📧 Email Simulation (credentials not provided):
   To: abiralsujakhu121@gmail.com
   Subject: Data Pipeline Report - COMPLETED WITH ERRORS - 2026-02-12 12:03
   Body preview: 
    <html>
    <body>
    <h2>🔄 Data Pipeline Execution Report</h2>
    
    <h3>📊 Summary</h3>
    <ul>
        <li><strong>Status:</strong> COMPLETED WITH ERRORS</li>
        <li><strong>Execution ...

PIPELINE EXECUTION SUMMARY
CSV Pipeline: Success (12575 rows)
API Pipeline: Failed (0 rows)
Errors: 1
  - API Error: Database error: Error binding parameter 5: type 'dict' is not supported


## Verify Database and View Logs

In [7]:
# Verify database contents
def verify_database(db_name):
    """Check all tables in the database"""
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    
    # Get all tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    
    print("📊 DATABASE VERIFICATION")
    print("=" * 50)
    print(f"Database: {db_name}")
    print(f"Tables found: {len(tables)}\n")
    
    for table in tables:
        table_name = table[0]
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cursor.fetchone()[0]
        print(f"  📁 {table_name}: {count} rows")
        
        # Show sample data
        df_sample = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 3", conn)
        print(f"     Columns: {list(df_sample.columns)[:5]}...")
    
    conn.close()
    print("=" * 50)

verify_database(db_name)

📊 DATABASE VERIFICATION
Database: data_pipeline.db
Tables found: 4

  📁 retail_sales: 12575 rows
     Columns: ['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit']...
  📁 api_posts: 100 rows
     Columns: ['userid', 'id', 'title', 'body', 'api_fetch_timestamp']...
  📁 retail_sales_final: 12575 rows
     Columns: ['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit']...
  📁 api_users_final: 0 rows
     Columns: ['id', 'name', 'username', 'email', 'address']...


In [8]:
# View log file contents
print("📋 LOG FILE CONTENTS")
print("=" * 50)

try:
    with open(log_file, 'r') as f:
        logs = f.read()
        print(logs)
except FileNotFoundError:
    print("Log file not found - run the pipeline cells first")

print("=" * 50)

📋 LOG FILE CONTENTS
2026-02-12 12:03:16,475 - DataPipeline - INFO - Logging configured successfully!
2026-02-12 12:03:16,539 - DataPipeline - INFO - ==================================================
2026-02-12 12:03:16,543 - DataPipeline - INFO - TASK 1: CSV to Database Pipeline Started
2026-02-12 12:03:16,547 - DataPipeline - INFO - ==================================================
2026-02-12 12:03:16,552 - DataPipeline - INFO - Starting to load CSV data from: c:\Users\mausa\Desktop\5thsem\adv_py\lab4\retail_store_sales.csv
2026-02-12 12:03:16,683 - DataPipeline - INFO - Successfully loaded 12575 rows and 11 columns
2026-02-12 12:03:16,686 - DataPipeline - DEBUG - Columns: ['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Discount Applied']
2026-02-12 12:03:16,716 - DataPipeline - INFO - Starting data transformation...
2026-02-12 12:03:16,731 - DataPipeline - INFO - Total missing valu

## Optional: Send Real Email

To actually send emails, you need to:
1. **Enable 2-Step Verification** on your Gmail account
2. **Generate an App Password** at https://myaccount.google.com/apppasswords
3. **Update the credentials** in the cell below and run it

In [10]:
# Send real email (update with your credentials)
# Uncomment and update the lines below to send actual emails

SENDER_EMAIL = "abiralsujakhu121@gmail.com"
SENDER_PASSWORD = "yhev pgxe bepd oqwe"  # 16-character App Password from Google

email_subject = "Test: Data Pipeline Notification"
email_body = """
<html>
<body>
<h2>✅ Data Pipeline Completed Successfully!</h2>
<p>This is a test notification from your Data Pipeline system.</p>
<p>Database has been updated with new data.</p>
</body>
</html>
"""

try:
    send_email_notification(
        recipient_email="abiralsujakhu121@gmail.com",
        subject=email_subject,
        body=email_body,
        sender_email=SENDER_EMAIL,
        sender_password=SENDER_PASSWORD
    )
    print("✅ Email sent successfully!")
except Exception as e:
    print(f"❌ Failed to send email: {e}")

print("💡 To send real emails, uncomment the code above and add your Gmail credentials")

2026-02-12 12:09:12,123 - DataPipeline - INFO - Preparing to send email notification to: abiralsujakhu121@gmail.com
2026-02-12 12:09:12,158 - DataPipeline - INFO - Connecting to SMTP server...
2026-02-12 12:09:16,802 - DataPipeline - INFO - Email sent successfully to abiralsujakhu121@gmail.com


✅ Email sent successfully!
💡 To send real emails, uncomment the code above and add your Gmail credentials
